In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/model_table.csv', parse_dates=['snapshot_date', 'signup_date'])

print(df.shape)
df.head()



Matplotlib is building the font cache; this may take a moment.


(4100, 47)


,account_id,signup_date,snapshot_date,churn_90d,industry,country,referral_source,tenure_days,n_subs,n_active_subs,...,n_closed,avg_resolution_hours,avg_first_response_min,avg_satisfaction,n_rated,escalations,rated_share,n_past_churns,days_since_last_churn,usage_trend
0,A-43a9e3,2023-08-17,2023-09-01,0,FinTech,IN,other,15,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0
1,A-43a9e3,2023-08-17,2023-10-01,0,FinTech,IN,other,45,1.0,1.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0
2,A-43a9e3,2023-08-17,2023-11-01,0,FinTech,IN,other,76,1.0,1.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,9.0
3,A-43a9e3,2023-08-17,2023-12-01,0,FinTech,IN,other,106,3.0,3.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,-9.0
4,A-43a9e3,2023-08-17,2024-01-01,0,FinTech,IN,other,137,3.0,3.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN,16.0


In [2]:
#Csv ye çevirirken tarih sütunları tekrardan string formatına dönüşür. Bu yüzden tekrardan date_time a dönüştürdüm

In [3]:
id_cols = ['account_id', 'snapshot_date', 'signup_date']
target_col = 'churn_90d'
cat_cols = ['industry', 'country', 'referral_source', 'latest_plan', 'latest_billing', 'latest_is_trial']
num_cols = [c for c in df.columns if c not in id_cols + [target_col] + cat_cols]

# Kategoriklerde boşluk = henüz abonelik yok; bunu ayrı bir kategori yapalım
df[cat_cols] = df[cat_cols].fillna('none').astype(str)

print(f"{len(num_cols)} numeric, {len(cat_cols)} categorical features")
print(num_cols)

37 numeric, 6 categorical features
['tenure_days', 'n_subs', 'n_active_subs', 'active_mrr', 'active_seats', 'n_upgrades', 'n_downgrades', 'days_since_last_sub_start', 'latest_seats', 'latest_mrr', 'usage_count_30d', 'usage_secs_30d', 'errors_30d', 'active_days_30d', 'n_features_30d', 'beta_events_30d', 'usage_count_90d', 'usage_secs_90d', 'errors_90d', 'active_days_90d', 'n_features_90d', 'beta_events_90d', 'usage_count_prev30d', 'days_since_last_usage', 'tickets_total', 'tickets_90d', 'open_tickets', 'n_closed', 'avg_resolution_hours', 'avg_first_response_min', 'avg_satisfaction', 'n_rated', 'escalations', 'rated_share', 'n_past_churns', 'days_since_last_churn', 'usage_trend']


In [4]:
#TRAIN-TEST Ayrımı

HORIZON_DAYS = 90
TEST_START = pd.Timestamp('2024-08-01')
TRAIN_END = TEST_START - pd.Timedelta(days=HORIZON_DAYS)

train = df[df['snapshot_date'] <= TRAIN_END]
test = df[df['snapshot_date'] >= TEST_START]
gap = df[(df['snapshot_date'] > TRAIN_END) & (df['snapshot_date'] < TEST_START)]

for name, part in [('train', train), ('gap', gap), ('test', test)]:
    print(f"{name:5}  snapshots={part['snapshot_date'].nunique():2}  "
          f"rows={len(part):5}  positive rate={part[target_col].mean():.1%}  "
          f"({part['snapshot_date'].min().date()} → {part['snapshot_date'].max().date()})")

X_train, y_train = train[num_cols + cat_cols], train[target_col]
X_test, y_test = test[num_cols + cat_cols], test[target_col]

train  snapshots=11  rows= 2236  positive rate=18.6%  (2023-07-01 → 2024-05-01)
gap    snapshots= 2  rows=  675  positive rate=25.6%  (2024-06-01 → 2024-07-01)
test   snapshots= 3  rows= 1189  positive rate=28.1%  (2024-08-01 → 2024-10-01)


In [5]:
#Değerlendirelim
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(name, y_true, scores, top_frac=0.10):
    y = np.asarray(y_true)
    s = np.asarray(scores)
    n_top = int(len(s) * top_frac)
    top_idx = np.argsort(-s)[:n_top]

    precision_top = y[top_idx].mean()
    return {
        'model': name,
        'roc_auc': roc_auc_score(y, s),
        'pr_auc': average_precision_score(y, s),
        'precision_top10': precision_top,
        'recall_top10': y[top_idx].sum() / y.sum(),
        'lift_top10': precision_top / y.mean(),
    }

results = []

In [6]:
rng = np.random.default_rng(42)
results.append(evaluate('baseline: random', y_test, rng.random(len(y_test))))

# 2) Tek feature: yeni hesaplar daha riskli
results.append(evaluate('baseline: newer accounts riskier', y_test, -X_test['tenure_days']))

pd.DataFrame(results).round(3)

,model,roc_auc,pr_auc,precision_top10,recall_top10,lift_top10
0,baseline: random,0.466,0.258,0.220,0.078,0.784
1,baseline: newer accounts riskier,0.646,0.439,0.551,0.195,1.961


In [7]:
#Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale', StandardScaler()),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

logreg = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

logreg.fit(X_train, y_train)

lr_train_scores = logreg.predict_proba(X_train)[:, 1]
lr_test_scores = logreg.predict_proba(X_test)[:, 1]

print(f"train ROC-AUC: {roc_auc_score(y_train, lr_train_scores):.3f}")
results.append(evaluate('logistic regression', y_test, lr_test_scores))

pd.DataFrame(results).round(3)

train ROC-AUC: 0.652


,model,roc_auc,pr_auc,precision_top10,recall_top10,lift_top10
0,baseline: random,0.466,0.258,0.220,0.078,0.784
1,baseline: newer accounts riskier,0.646,0.439,0.551,0.195,1.961
2,logistic regression,0.574,0.338,0.356,0.126,1.267


In [8]:
#İki hipotezi test edelim
# Hipotez B: tenure train döneminde de güçlü bir sinyal mi?
print(f"tenure baseline — train ROC-AUC: {roc_auc_score(y_train, -X_train['tenure_days']):.3f}")
print(f"tenure baseline — test ROC-AUC:  {roc_auc_score(y_test, -X_test['tenure_days']):.3f}")

tenure baseline — train ROC-AUC: 0.576
tenure baseline — test ROC-AUC:  0.646


In [9]:
# Hipotez A: daha az feature'la logistic regression
def make_logreg(num, cat):
    prep = ColumnTransformer([
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median', add_indicator=True)),
            ('scale', StandardScaler()),
        ]), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])
    return Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ])

small_num = ['tenure_days', 'active_mrr', 'n_subs', 'n_past_churns', 'days_since_last_churn']
small_cat = ['industry', 'referral_source', 'latest_plan']

logreg_small = make_logreg(small_num, small_cat)
logreg_small.fit(X_train, y_train)

print(f"train ROC-AUC: {roc_auc_score(y_train, logreg_small.predict_proba(X_train)[:, 1]):.3f}")
results.append(evaluate('logistic regression (small)', y_test, logreg_small.predict_proba(X_test)[:, 1]))

pd.DataFrame(results).round(3)

train ROC-AUC: 0.602


,model,roc_auc,pr_auc,precision_top10,recall_top10,lift_top10
0,baseline: random,0.466,0.258,0.220,0.078,0.784
1,baseline: newer accounts riskier,0.646,0.439,0.551,0.195,1.961
2,logistic regression,0.574,0.338,0.356,0.126,1.267
3,logistic regression (small),0.621,0.386,0.432,0.153,1.539


In [10]:
#LIGHTGBM
from lightgbm import LGBMClassifier

def to_lgb(X):
    X = X.copy()
    for c in cat_cols:
        X[c] = X[c].astype(pd.CategoricalDtype(sorted(df[c].unique())))
    return X

X_train_lgb, X_test_lgb = to_lgb(X_train), to_lgb(X_test)

lgb_params = dict(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)

# Tüm feature'larla
lgb_full = LGBMClassifier(**lgb_params)
lgb_full.fit(X_train_lgb, y_train)
print(f"full  — train ROC-AUC: {roc_auc_score(y_train, lgb_full.predict_proba(X_train_lgb)[:, 1]):.3f}")
results.append(evaluate('lightgbm (full)', y_test, lgb_full.predict_proba(X_test_lgb)[:, 1]))

# Küçük feature setiyle
small_cols = small_num + small_cat
lgb_small = LGBMClassifier(**lgb_params)
lgb_small.fit(X_train_lgb[small_cols], y_train)
print(f"small — train ROC-AUC: {roc_auc_score(y_train, lgb_small.predict_proba(X_train_lgb[small_cols])[:, 1]):.3f}")
results.append(evaluate('lightgbm (small)', y_test, lgb_small.predict_proba(X_test_lgb[small_cols])[:, 1]))

pd.DataFrame(results).round(3)

full  — train ROC-AUC: 0.993
small — train ROC-AUC: 0.945


,model,roc_auc,pr_auc,precision_top10,recall_top10,lift_top10
0,baseline: random,0.466,0.258,0.220,0.078,0.784
1,baseline: newer accounts riskier,0.646,0.439,0.551,0.195,1.961
2,logistic regression,0.574,0.338,0.356,0.126,1.267
3,logistic regression (small),0.621,0.386,0.432,0.153,1.539
4,lightgbm (full),0.537,0.300,0.331,0.117,1.177
5,lightgbm (small),0.574,0.343,0.381,0.135,1.358


In [11]:
#Hangi featurelar etkili
feature_names = logreg_small.named_steps['prep'].get_feature_names_out()
coefs = logreg_small.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'odds_ratio': np.exp(coefs),
}).sort_values('coef', key=abs, ascending=False)

coef_df.round(3)

,feature,coef,odds_ratio
0,num__tenure_days,-0.315,0.730
13,cat__referral_source_organic,0.257,1.294
1,num__active_mrr,-0.254,0.776
5,num__missingindicator_days_since_last_churn,-0.221,0.802
3,num__n_past_churns,-0.217,0.805
2,num__n_subs,0.171,1.187
6,cat__industry_Cybersecurity,0.156,1.169
15,cat__referral_source_partner,-0.154,0.857
17,cat__latest_plan_Enterprise,0.140,1.150
9,cat__industry_FinTech,-0.127,0.881


## Modelleme sonuçları

**Kurulum**

Test seti olarak son 3 snapshot'ı aldım (2024-08 ile 2024-10 arası, 1.189 satır). Train 2023-07 ile 2024-05 arası (11 snapshot, 2.236 satır). Aradaki iki snapshot'ı (Haziran ve Temmuz) kullanmadım, çünkü train'in etiket penceresi 90 gün ileriye bakıyor. Bu boşluk olmasa train etiketleri test dönemindeki churn'leri görmüş olurdu.

Pozitif oranı train'de %18.6, test'te %28.1. Model daha sakin bir dönemden öğrenip daha yoğun bir dönemde test ediliyor.

**Sonuçlar (test seti)**

| Model | ROC-AUC | PR-AUC | Lift (top %10) |
|---|---|---|---|
| Rastgele | 0.466 | 0.258 | 0.78 |
| Sadece tenure ("yeni hesaplar daha riskli") | **0.646** | **0.439** | **1.96** |
| Logistic regression (tüm feature'lar) | 0.574 | 0.338 | 1.27 |
| Logistic regression (küçük set) | 0.621 | 0.386 | 1.54 |
| LightGBM (tüm feature'lar) | 0.537 | 0.300 | 1.18 |
| LightGBM (küçük set) | 0.574 | 0.343 | 1.36 |

Rastgele skorlar bile 0.466 verdi, yani bu test setinde ±0.03-0.04'lük farkları tesadüften ayırt edemiyorum. Bunu modelleri karşılaştırırken dikkate aldım.

**Ne oldu?**

En iyi sonuç tek bir kuraldan geldi: hesapları yaşına göre sıralamak. En riskli %10'luk listenin %55'i gerçekten churn ediyor, rastgele seçime göre neredeyse iki kat isabet.

Logistic regression bu kuralı geçemedi. İki sebep buldum:

- Tüm feature'larla model 0.574'te kaldı, kullanım ve destek feature'larını çıkarınca 0.621'e çıktı. Bu feature'lar churn ile ilişkisiz (01_eda'daki tarih tutarsızlığıyla uyumlu) ve model train'de bunlarda tesadüfi örüntüler öğreniyordu
- Tenure'ın etkisi zamanla güçlenmiş: tek başına train'de 0.576, test'te 0.646. Model, ilişkinin daha zayıf olduğu dönemden öğrendiği için tenure'a olması gerekenden az ağırlık veriyor

LightGBM daha da kötü: train'de 0.993, test'te 0.537. Ağaçları küçük tutmama rağmen ezberledi. Bence sebebi aynı hesabın train'de 11 satıra kadar tekrar etmesi; model sektör, kanal ve MRR kombinasyonlarından hesabı tanıyıp etiketlerini hatırlayabiliyor. Logistic regression doğrusal olduğu için bunu yapamıyor, train ve test skorları birbirine yakın (0.602 ve 0.621).

LightGBM'i test sonucuna bakarak yeniden ayarlamadım, çünkü bu test setine uyum sağlamak olurdu. Sinyal bu kadar zayıfken train içinde ayrı bir doğrulama setiyle ayarlamanın da sonucu değiştireceğini düşünmüyorum.

**Sonuç**

Bu veride churn'ü en iyi açıklayan şey hesabın yaşı. Diğer feature'lar ona anlamlı bir katkı yapmıyor, karmaşık modeller de işe yaramıyor. Gerçek bir senaryoda ilk önerim basit bir kural olurdu: yeni hesapları öncelikli takibe almak. Kullanım ve destek verisinin bu veri setinde churn ile ilişkisiz olması, sentetik verinin bir sınırlaması gibi görünüyor. Gerçek bir üründe bu iki kaynak genelde en güçlü sinyalleri taşır.

**Hangi feature'lar etkili? (logistic regression, küçük set)**

Sayısal feature'lar ölçeklendiği için katsayılar 1 standart sapmalık değişim başına etkiyi gösteriyor.

Güvendiğim üç sinyal, hem tanımlayıcı tablolarda hem modelde aynı yönde:
- Hesap yaşı en güçlü etki: yaş 1 standart sapma artınca churn odds'u ~%27 düşüyor
- Organik kanaldan gelen hesaplar daha riskli (odds ~%29 daha yüksek)
- MRR'ı yüksek hesaplar daha az riskli (odds ~%22 daha düşük)

Güvenmediklerim:
- Geçmiş churn feature'ları birbiriyle çelişiyor gibi: "hiç churn etmemiş" riski düşürüyor ama geçmiş churn sayısının katsayısı da negatif. Bu iki feature neredeyse aynı bilgiyi taşıyor ve ikisi de hesap yaşıyla ilişkili, bu yüzden model etkiyi aralarında rastgele paylaştırıyor olabilir. n_subs için de aynı durum geçerli
- Sektör ve plan katsayıları küçük ve tanımlayıcı tablolarla çelişiyor (Cybersecurity tabloda en düşük churn oranına sahipti ama modelde riski artırıyor). Gürültü düzeyinde olduklarını düşünüyorum